# Notebook 4 — Final evaluation on the test set

**Computer Vision CW1 — Diabetic Retinopathy Stage Detection**
Mohamed Shafran · NIBM (BSCCOMP24.2P)

Notebook 3 chose one final model using the **validation** split only. This notebook measures that model **once** on the **test** split: 525 images it has never seen and that played no part in any decision.

**The one-time rule.** Nothing here trains the model or changes a setting — not the model, not the preprocessing variant, not the 0.5 threshold of the decision rule. If a result is disappointing, it is reported and discussed, never "fixed" by going back and tuning on the test images. Tuning on the test set would make its scores optimistic, which is data leakage.

| Section | What it answers |
|---|---|
| 1–2 | Which model is evaluated, and does it reproduce its notebook 3 validation score? |
| 3–6 | Test results: 5 stages, DR yes/no, per stage, confusion matrix, ROC curve, uncertainty |
| 7 | Did the final model overfit during training? |
| 8 | Error analysis: how far off, which stages, which cameras, which image qualities |
| 9 | Grad-CAM: where in the image the model looks |

**How to run:**
1. *Accelerator*: **None** (CPU). This notebook only makes predictions, which a CPU does in a few minutes, so no GPU hours are used.
2. *Input*: the outputs of notebooks 2 **and** 3 (*Add Input → Your Work → Notebooks → dr-02-preprocess*, then *dr-03-train-experiments*).
3. **Save Version → Save & Run All**. It takes roughly 10–20 minutes.

In [ ]:
# ============================================================
# Setup: imports, folders, inputs, shared style
# ============================================================
import os, sys, json, zipfile, warnings, importlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import cv2
import tensorflow as tf
import keras
from sklearn.metrics import (classification_report, confusion_matrix, precision_recall_fscore_support,
                             precision_score, recall_score, f1_score, accuracy_score, cohen_kappa_score,
                             roc_auc_score, roc_curve)

OUT_DIR = Path("/kaggle/working")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUT_DIR)                                      # dr_gradcam.py is written here (section 9)

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
STAGE_COLORS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK_SOFT, MUTED, GRID, FLAG = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#d03b3b"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK_SOFT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "font.size": 10, "figure.dpi": 110, "savefig.bbox": "tight",
})

def tidy(ax, grid_axis="y"):
    """Remove chart junk: drop the top/right frame, put a hairline grid behind the data."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
    return ax

def find_file(name, root="/kaggle/input"):
    """Search the attached inputs for a file, wherever Kaggle mounted it."""
    for path, dirs, files in os.walk(root):
        if name in files:
            return Path(path) / name
    raise FileNotFoundError(f"{name} not found under {root} - are the outputs of notebooks 2 and 3 attached?")

# notebook 2's output: the split (with the camera columns), the image archives, the preprocessing module
splits = pd.read_csv(find_file("splits.csv"))
sys.path.insert(0, str(find_file("dr_preprocessing.py").parent))
import dr_preprocessing as prep

# notebook 3's output: the final model, its settings, and the training module (for the shared metrics)
NB3_DIR = find_file("best_config.json").parent
sys.path.insert(0, str(NB3_DIR))
import dr_training as T
config = json.loads((NB3_DIR / "best_config.json").read_text())
VARIANT, SIZE, BACKBONE = config["variant"], int(config["image_size"]), config["backbone"]
model = keras.models.load_model(NB3_DIR / "best_model.keras", compile=False)   # compile=False: no training here

print("GPU:", [g.name for g in tf.config.list_physical_devices("GPU")] or "none - the CPU is enough here")
print("images:", len(splits), "| test:", int((splits["split"] == "test").sum()))

## 1. The model being evaluated

The table is notebook 3's summary: one row per configuration, validation scores averaged over its seeds. The final model is the best seed of the winning configuration.

In [ ]:
summary_nb3 = pd.read_csv(NB3_DIR / "experiment_summary.csv", index_col=0)
columns = [c for c in ["experiment", "backbone", "variant", "balance", "ft_lr", "runs", "macro_f1", "qwk",
                       "dr_accuracy"] if c in summary_nb3]
shown = summary_nb3[columns].round(3)
if "ft_lr" in shown:
    shown["ft_lr"] = summary_nb3["ft_lr"].map("{:g}".format)      # 0.0001 would round to 0.0
print(shown.to_string())

print(f"\nfinal model: {config['name']} = {BACKBONE} at {SIZE} px, variant {VARIANT} "
      f"({prep.VARIANTS[VARIANT]}), balancing '{config['balance']}', fine-tuning lr {config['fine_tune_lr']:g}")
print(f"kept epoch {config['best_epoch']} (phase 1: {config['epochs_phase1']} epochs, phase 2: "
      f"{config['epochs_phase2']} epochs) | {model.count_params() / 1e6:.1f} M parameters")
print(f"decision rule: {config['decision_rule']}")

## 2. First, check the pipeline on the validation split

Before the test images are touched, the validation images are prepared exactly as in notebook 3 (decoded from notebook 2's archive, resized with area interpolation) and passed through the saved model. The answers must match the ones notebook 3 saved. If they do not, something differs — the images, the resizing or the model file — and every test number would be wrong, so the notebook stops.

This also shows that the saved model file works on a CPU, which is how the prototype app will run it on the Mac.

In [ ]:
def load_images(ids, variant=VARIANT, size=SIZE):
    """Decode images from notebook 2's archive of one variant and resize them exactly as notebook 3
    did (area interpolation). size=None keeps the 512 x 512 original, for display."""
    with zipfile.ZipFile(find_file(f"preprocessed_{variant}.zip")) as archive:
        blobs = [archive.read(f"{c}.png") for c in ids]

    def decode(blob):
        img = cv2.cvtColor(cv2.imdecode(np.frombuffer(blob, np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if size and size != img.shape[0]:
            img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
        return img

    with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
        return np.stack(list(pool.map(decode, blobs)))


val = splits[splits["split"] == "val"].reset_index(drop=True)
X_val = load_images(val["id_code"])
val_probs = model.predict(X_val.astype("float32"), batch_size=32, verbose=0)
val_answer = T.two_step(val_probs)
del X_val

saved = pd.read_csv(NB3_DIR / "runs" / config["name"] / "val_predictions.csv").set_index("id_code")
same = float((saved.loc[val["id_code"], "pred"].to_numpy() == val_answer).mean())
now = T.scores(val["diagnosis"], val_answer)
print(f"answers identical to notebook 3: {same:.1%} of {len(val)} validation images")
print(f"validation macro-F1 {now['macro_f1']:.3f} (notebook 3: {config['macro_f1']:.3f}) | "
      f"QWK {now['qwk']:.3f} ({config['qwk']:.3f}) | DR accuracy {now['dr_accuracy']:.3f} ({config['dr_accuracy']:.3f})")
# a GPU and a CPU round numbers slightly differently, so an image or two right at the 0.5 threshold may flip
assert same >= 0.98, "the saved model does not reproduce notebook 3 - stop here and investigate"

## 3. Predict the test set — once

Every test image gets five probabilities, and the **two-step rule** turns them into the system's answer: *DR present* if the probabilities of stages 1–4 add up to 0.5 or more, then the most likely of stages 1–4. The answers are saved to `test_predictions.csv`, so every analysis below reads the same saved predictions.

In [ ]:
test = splits[splits["split"] == "test"].reset_index(drop=True)
X_test = load_images(test["id_code"])
probs = model.predict(X_test.astype("float32"), batch_size=32, verbose=0)

true = test["diagnosis"].to_numpy()
pred = T.two_step(probs)                         # the system's answer (stage 0-4)
p_dr = probs[:, 1:].sum(axis=1)                  # the probability of "DR present"

predictions = test[["id_code", "camera", "camera_group"]].copy()
predictions["true"], predictions["pred"], predictions["p_dr"] = true, pred, p_dr
for k in range(5):
    predictions[f"p{k}"] = probs[:, k]
predictions["highest_probability"] = probs.argmax(axis=1)
predictions.to_csv(OUT_DIR / "test_predictions.csv", index=False)

print("test images per stage:", {CLASS_NAMES[k]: int(n) for k, n in enumerate(np.bincount(true, minlength=5))})
print("answers per stage:    ", {CLASS_NAMES[k]: int(n) for k, n in enumerate(np.bincount(pred, minlength=5))})

## 4. Headline results

The metrics from notebooks 2 and 3 (accuracy, macro-F1, QWK, DR yes/no accuracy, sensitivity, specificity), plus:

- **Precision** of a stage — of the images the model *called* that stage, the share that really are. **Recall** — of the images that really *are* that stage, the share the model found. **F1** combines the two.
- **Macro** average — every stage counts equally. **Weighted** average — every *image* counts equally, so No DR dominates it.
- **DR precision** — of the eyes flagged as DR, the share that really have DR.
- **ROC-AUC** *(in your Siamese-network practical)* — how well the probability of DR separates diseased from healthy eyes over *every* possible threshold: 0.5 is chance, 1.0 is perfect. The two-step rule uses one threshold, 0.5.
- **Referable DR** *(additional knowledge)* — screening programmes often refer a patient to an eye specialist at *moderate DR or worse* (stage 2 or higher). Sensitivity and specificity at that cut show how the model would work as a referral filter.

The same metrics are shown for the final model on validation (for comparison — did the test scores drop?) and for the two models that never look at the retina, now scored on the test split. The camera-only model uses the most common stage of each image size in the **training** split.

In [ ]:
def full_scores(true, pred, p_dr=None):
    """All the metrics of this notebook for one set of answers."""
    true, pred = np.asarray(true), np.asarray(pred)
    s = T.scores(true, pred)                                  # the metrics of notebooks 2 and 3
    dr_true, dr_pred = true > 0, pred > 0
    ref_true, ref_pred = true >= 2, pred >= 2
    stages = list(range(5))
    s.update({
        "macro_precision": precision_score(true, pred, labels=stages, average="macro", zero_division=0),
        "macro_recall": recall_score(true, pred, labels=stages, average="macro", zero_division=0),
        "weighted_f1": f1_score(true, pred, labels=stages, average="weighted", zero_division=0),
        "dr_precision": precision_score(dr_true, dr_pred, zero_division=0),
        "dr_f1": f1_score(dr_true, dr_pred, zero_division=0),
        "referable_sensitivity": recall_score(ref_true, ref_pred, zero_division=0),
        "referable_specificity": recall_score(~ref_true, ~ref_pred, zero_division=0),
        "dr_roc_auc": roc_auc_score(dr_true, p_dr) if p_dr is not None else np.nan,
    })
    return {k: float(v) for k, v in s.items()}


# the camera-only model, built from the TRAINING split exactly as in notebook 2
train = splits[splits["split"] == "train"]
stage_for_camera = train.groupby("camera")["diagnosis"].agg(lambda s: s.value_counts().idxmax())
most_common_stage = int(train["diagnosis"].value_counts().idxmax())
camera_guess = test["camera"].map(stage_for_camera).fillna(most_common_stage).astype(int).to_numpy()
predictions["camera_only"] = camera_guess

results = pd.DataFrame({
    "final model (test)": full_scores(true, pred, p_dr),
    "final model (validation)": full_scores(val["diagnosis"], val_answer, val_probs[:, 1:].sum(axis=1)),
    "camera only (test)": full_scores(true, camera_guess),
    "always No DR (test)": full_scores(true, np.zeros_like(true)),
})
results.index.name = "metric"
results.to_csv(OUT_DIR / "test_results.csv")
print(results.round(3).to_string())

### How certain are these numbers? *(additional knowledge: bootstrap confidence intervals)*

The test set is one sample of 525 images, with only a few dozen for the rarest stages. A different sample would give somewhat different scores. The **bootstrap** estimates how much: it draws 525 images *with replacement* from the test predictions (some twice, some not at all), recomputes every metric, and repeats this 2,000 times. The middle 95 % of the 2,000 results is the **95 % confidence interval**. A wide interval means the number is uncertain. Nothing is re-predicted, and the model is not involved.

In [ ]:
rng = np.random.default_rng(42)
N_BOOT = 2000
rows = []
for _ in range(N_BOOT):
    i = rng.integers(0, len(true), len(true))            # a resample of the test set, with replacement
    t, p = true[i], pred[i]
    precision, recall, f1, _ = precision_recall_fscore_support(t, p, labels=list(range(5)), zero_division=0)
    row = T.scores(t, p)
    both = 0 < (t > 0).sum() < len(t)                      # ROC-AUC needs DR and No DR in the resample
    row["dr_roc_auc"] = roc_auc_score(t > 0, p_dr[i]) if both else np.nan
    row.update({f"{name}_{k}": values[k] for name, values in (("precision", precision), ("recall", recall),
                                                               ("f1", f1)) for k in range(5)})
    rows.append(row)
boot = pd.DataFrame(rows)
ci = boot.quantile([0.025, 0.975]).T
ci.columns = ["ci_low", "ci_high"]

headline = ["accuracy", "macro_f1", "qwk", "dr_accuracy", "dr_sensitivity", "dr_specificity", "dr_roc_auc"]
ci_table = pd.DataFrame({"test": results.loc[headline, "final model (test)"],
                         "ci_low": ci.loc[headline, "ci_low"], "ci_high": ci.loc[headline, "ci_high"]})
ci.to_csv(OUT_DIR / "test_bootstrap_ci.csv")
print("final model on the test split, with 95% bootstrap confidence intervals:")
print(ci_table.round(3).to_string())

## 5. Per-stage results

The **classification report** lists precision, recall and F1 for every stage, with the number of test images of that stage (*support*). The **confusion matrix** shows where the answers go: rows are the true stages, columns the model's answers, and each cell gives the count and the share of its row. The diagonal holds the correct answers. Because the stages are ordered, a mistake next to the diagonal (for example Mild called Moderate) is much less serious than one far from it.

In [ ]:
print(classification_report(true, pred, labels=list(range(5)), target_names=CLASS_NAMES, digits=3, zero_division=0))
report = pd.DataFrame(classification_report(true, pred, labels=list(range(5)), target_names=CLASS_NAMES,
                                            output_dict=True, zero_division=0)).T
report.to_csv(OUT_DIR / "test_classification_report.csv")

# Figure 26 - confusion matrix on the test split
matrix = confusion_matrix(true, pred, labels=range(5))
row_share = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(6.6, 5.6))
ax.imshow(row_share, cmap="Blues", vmin=0, vmax=1)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{matrix[i, j]}\n{100 * row_share[i, j]:.0f}%", ha="center", va="center",
                fontsize=9, color="white" if row_share[i, j] > 0.55 else INK)
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels([f"{i} {CLASS_NAMES[i]}" for i in range(5)], rotation=30, ha="right")
ax.set_yticklabels([f"{i} {CLASS_NAMES[i]}" for i in range(5)])
ax.set_xlabel("Predicted stage")
ax.set_ylabel("True stage")
ax.set_title(f"Test confusion matrix - accuracy {results.loc['accuracy', 'final model (test)']:.1%}")
plt.savefig(FIG_DIR / "fig26_test_confusion.png")
plt.show()

In [ ]:
# Figure 27 - precision, recall and F1 of every stage, with 95% bootstrap intervals
train_counts = np.bincount(train["diagnosis"], minlength=5)
support = np.bincount(true, minlength=5)
precision, recall, f1, _ = precision_recall_fscore_support(true, pred, labels=list(range(5)), zero_division=0)

fig, ax = plt.subplots(figsize=(12, 4.6))
x, width = np.arange(5), 0.26
for k, (name, values, colour) in enumerate([("precision", precision, "#86b6ef"), ("recall", recall, BLUE),
                                            ("f1", f1, "#104281")]):
    low = ci.loc[[f"{name}_{s}" for s in range(5)], "ci_low"].to_numpy()
    high = ci.loc[[f"{name}_{s}" for s in range(5)], "ci_high"].to_numpy()
    xs = x + (k - 1) * width
    ax.bar(xs, values, width=width - 0.03, color=colour, label=name.capitalize() if name != "f1" else "F1")
    ax.errorbar(xs, values, yerr=[np.maximum(values - low, 0), np.maximum(high - values, 0)], fmt="none",
                ecolor=INK_SOFT, capsize=2.5, linewidth=0.9)
    if name == "f1":                                          # print the F1 values, above their whiskers
        for xi, v, h in zip(xs, values, high):
            ax.text(xi, max(v, h) + 0.02, f"{v:.2f}", ha="center", fontsize=8.5, color=INK)
macro = results.loc["macro_f1", "final model (test)"]
ax.axhline(macro, color=ORANGE, linestyle="--", linewidth=1.2)
ax.text(-0.45, macro + 0.015, f"macro-F1 {macro:.2f}", ha="left", fontsize=8.5, color=ORANGE)
ax.set_xticks(x)
ax.set_xticklabels([f"{i} {CLASS_NAMES[i]}\ntest n={support[i]} | train n={train_counts[i]}" for i in range(5)])
ax.set_ylim(0, 1.18)
ax.set_ylabel("Score on the test split")
ax.set_title("Per-stage precision, recall and F1 (whiskers: 95% bootstrap interval)")
ax.legend(frameon=False, labelcolor=INK_SOFT, ncols=3, loc="upper right", fontsize=9)
tidy(ax)
plt.savefig(FIG_DIR / "fig27_test_per_stage.png")
plt.show()

per_stage = pd.DataFrame({"training_images": train_counts, "test_images": support, "precision": precision,
                          "recall": recall, "f1": f1}, index=CLASS_NAMES)
per_stage.to_csv(OUT_DIR / "test_per_stage.csv")
print(per_stage.round(3).to_string())

## 6. DR yes/no: the ROC curve and the operating point

The first question the system answers is *does this eye have DR?* The ROC curve shows every trade-off between catching diseased eyes (sensitivity, up) and wrongly flagging healthy ones (1 − specificity, right) that some threshold on the probability of DR would give. The orange dot is the threshold the system actually uses, 0.5, fixed in the design before any test result was seen. The red cross is the camera-only model, for comparison.

In screening, a missed diseased eye (a *false negative*) is usually worse than a false alarm, which only leads to a second look by a specialist.

In [ ]:
dr_true, dr_pred = true > 0, pred > 0
fpr, tpr, _ = roc_curve(dr_true, p_dr)
auc = results.loc["dr_roc_auc", "final model (test)"]
sens, spec = results.loc["dr_sensitivity", "final model (test)"], results.loc["dr_specificity", "final model (test)"]
cam_sens, cam_spec = results.loc["dr_sensitivity", "camera only (test)"], results.loc["dr_specificity", "camera only (test)"]

# Figure 28 - ROC curve (left) and the 2 x 2 yes/no table (right)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), gridspec_kw={"width_ratios": [1.2, 1]})
axes[0].plot(fpr, tpr, color=BLUE, linewidth=2, label=f"final model (AUC {auc:.3f})")
axes[0].plot([0, 1], [0, 1], color=MUTED, linestyle=":", linewidth=1.2, label="chance (AUC 0.5)")
axes[0].scatter([1 - spec], [sens], s=80, color=ORANGE, zorder=3,
                label=f"threshold 0.5: sensitivity {sens:.2f}, specificity {spec:.2f}")
axes[0].scatter([1 - cam_spec], [cam_sens], s=80, marker="X", color=FLAG, zorder=3,
                label=f"camera only: sensitivity {cam_sens:.2f}, specificity {cam_spec:.2f}")
axes[0].set_xlim(-0.02, 1.02)
axes[0].set_ylim(-0.02, 1.02)
axes[0].set_xlabel("False-positive rate (1 - specificity)")
axes[0].set_ylabel("Sensitivity (true-positive rate)")
axes[0].set_title("DR yes/no: ROC curve on the test split")
axes[0].legend(frameon=False, labelcolor=INK_SOFT, loc="lower right", fontsize=8.5)
tidy(axes[0], grid_axis="both")

yes_no = confusion_matrix(dr_true, dr_pred, labels=[False, True])
share = yes_no / np.maximum(yes_no.sum(axis=1, keepdims=True), 1)
axes[1].imshow(share, cmap="Blues", vmin=0, vmax=1)
names = [["true negative", "false positive"], ["false negative", "true positive"]]
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{names[i][j]}\n{yes_no[i, j]} ({100 * share[i, j]:.0f}%)", ha="center",
                     va="center", fontsize=10, color="white" if share[i, j] > 0.55 else INK)
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(["No DR", "DR"])
axes[1].set_yticklabels(["No DR", "DR"])
axes[1].set_xlabel("Model's answer")
axes[1].set_ylabel("True")
axes[1].set_title(f"{yes_no[1, 0]} of {yes_no[1].sum()} DR eyes missed")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig28_test_dr_yes_no.png")
plt.show()

## 7. Did the final model overfit?

*(The overfitting-gap plot from your augmentation practical.)* For every training epoch of the final model: training accuracy minus validation accuracy, and validation loss minus training loss. A gap that keeps growing means the model is memorising the training images instead of learning what generalises. The vertical dotted line marks the epoch whose weights were kept (the best validation macro-F1), so later epochs never reach the final model.

Read the gap with care: training accuracy is measured on *augmented* images with dropout switched on (and, with oversampling, on a balanced mix of stages), so it is lower than the model's accuracy on clean training images. The real gap is therefore somewhat larger than the one plotted.

In [ ]:
history = pd.read_csv(NB3_DIR / "runs" / config["name"] / "history.csv")
epochs = np.arange(1, len(history) + 1)
phase1_epochs = int((history["phase"] == 1).sum())

# Figure 29 - overfitting gap across both training phases
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
gaps = [("accuracy", history["accuracy"] - history["val_accuracy"], "Training minus validation accuracy"),
        ("loss", history["val_loss"] - history["loss"], "Validation minus training loss")]
for ax, (key, gap, label) in zip(axes, gaps):
    ax.bar(epochs, gap, color=[BLUE if g >= 0 else MUTED for g in gap], width=0.7)
    ax.axhline(0, color=INK_SOFT, linewidth=0.8)
    ax.axvline(phase1_epochs + 0.5, color=MUTED, linestyle="--", linewidth=1)
    ax.axvline(config["best_epoch"], color=ORANGE, linestyle=":", linewidth=1.5)
    ax.text(phase1_epochs + 0.6, ax.get_ylim()[1], " fine-tuning", va="top", fontsize=8.5, color=INK_SOFT)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Gap")
    ax.set_title(label)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))           # whole epochs only
    tidy(ax)
fig.suptitle(f"Overfitting gap of {config['name']} (dotted orange: the epoch that was kept)",
             fontsize=12, fontweight="bold", color=INK)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.savefig(FIG_DIR / "fig29_overfitting_gap.png")
plt.show()

best = config["best_epoch"] - 1
print(f"at the kept epoch {config['best_epoch']}: training accuracy {history['accuracy'][best]:.3f}, "
      f"validation accuracy {history['val_accuracy'][best]:.3f} | training loss {history['loss'][best]:.3f}, "
      f"validation loss {history['val_loss'][best]:.3f}")

## 8. Error analysis

### 8.1 How far off are the mistakes?

Stages are ordered, so a mistake can be *one stage away* (e.g. Moderate called Severe — often a hard borderline case even for graders) or *two or more stages away* (e.g. Proliferative called No DR — a serious miss).

In [ ]:
distance = np.abs(pred - true)
off = pd.crosstab(true, np.minimum(distance, 2)).reindex(index=range(5), columns=[0, 1, 2], fill_value=0)
off_share = off.div(off.sum(axis=1), axis=0).fillna(0)
wrong_total = int((distance > 0).sum())
one_away = int((distance == 1).sum())

# Figure 30 - how far off, for every true stage
fig, ax = plt.subplots(figsize=(10, 3.8))
left = np.zeros(5)
for col, label, colour in [(0, "correct", BLUE), (1, "one stage away", "#86b6ef"), (2, "two or more stages away", FLAG)]:
    widths = off_share[col].to_numpy() * 100
    ax.barh(range(5), widths, left=left, height=0.62, color=colour, edgecolor="#fcfcfb", linewidth=0.8, label=label)
    for yi, (w, l, n) in enumerate(zip(widths, left, off[col])):
        if w >= 7:
            ax.text(l + w / 2, yi, f"{n}", ha="center", va="center", fontsize=9,
                    color="white" if colour != "#86b6ef" else INK)
    left += widths
ax.set_yticks(range(5))
ax.set_yticklabels([f"{i} {CLASS_NAMES[i]} (n={support[i]})" for i in range(5)])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Percent of the test images of that stage")
ax.set_title(f"{one_away} of the {wrong_total} mistakes ({one_away / max(wrong_total, 1):.0%}) are one stage away")
ax.legend(frameon=False, ncols=3, fontsize=9, labelcolor=INK_SOFT, loc="upper center", bbox_to_anchor=(0.5, -0.18))
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.savefig(FIG_DIR / "fig30_error_distance.png")
plt.show()

signed = pd.Series(pred - true).groupby(true).mean()
print("average (answer - true stage) per true stage; negative = the model under-grades:")
print({CLASS_NAMES[k]: round(float(v), 2) for k, v in signed.items()})

### 8.2 Does class imbalance show?

The table under figure 27 lists, for every stage, the number of **training** images next to its test recall and F1. If the imbalance hurts, the stages with the fewest training images will have the lowest recall — keeping in mind that some stages are also harder to see (a Mild case can hinge on a few tiny microaneurysms).

### 8.3 Does the model do more than recognise the camera?

Notebook 2 showed that the image size (a stand-in for the camera and clinic) predicts the stage surprisingly well. Below, the final model and the camera-only model are compared **within each camera**. Where a camera photographed almost only healthy eyes, both score high and the comparison says little. The informative cameras are those with a **mix** of stages. There, only a model that looks at the retina can do well.

In [ ]:
rows = {}
with warnings.catch_warnings():                 # QWK is undefined (NaN) for a camera with one stage only
    warnings.simplefilter("ignore")
    for camera, g in predictions.groupby("camera_group"):
        rows[camera] = {"images": len(g), "share_no_dr": float((g["true"] == 0).mean()),
                        "model_accuracy": accuracy_score(g["true"], g["pred"]),
                        "camera_only_accuracy": accuracy_score(g["true"], g["camera_only"]),
                        "model_dr_accuracy": accuracy_score(g["true"] > 0, g["pred"] > 0),
                        "camera_only_dr_accuracy": accuracy_score(g["true"] > 0, g["camera_only"] > 0),
                        "model_qwk": cohen_kappa_score(g["true"], g["pred"], weights="quadratic")}
by_camera = pd.DataFrame(rows).T.astype(float).sort_values("images", ascending=False)
by_camera.to_csv(OUT_DIR / "test_by_camera.csv")
print(by_camera.round(3).to_string())

# Figure 31 - the model against the camera-only model, camera by camera
fig, axes = plt.subplots(1, 2, figsize=(13, 0.62 * len(by_camera) + 1.8), sharey=True)
y = np.arange(len(by_camera))
for ax, (metric, label) in zip(axes, [("accuracy", "Stage accuracy"), ("dr_accuracy", "DR yes/no accuracy")]):
    model_v = by_camera[f"model_{metric}"].to_numpy(dtype=float)
    camera_v = by_camera[f"camera_only_{metric}"].to_numpy(dtype=float)
    ax.barh(y - 0.19, model_v, height=0.36, color=BLUE, label="final model")
    ax.barh(y + 0.19, camera_v, height=0.36, color=MUTED, label="camera only")
    for yi, a, b in zip(y, model_v, camera_v):
        ax.text(a + 0.01, yi - 0.19, f"{a:.2f}", va="center", fontsize=8, color=INK_SOFT)
        ax.text(b + 0.01, yi + 0.19, f"{b:.2f}", va="center", fontsize=8, color=INK_SOFT)
    ax.set_xlim(0, 1.1)
    ax.set_title(label)
    tidy(ax, grid_axis="x")
axes[0].set_yticks(y)
axes[0].set_yticklabels([f"{c}  (n={int(r.images)}, {r.share_no_dr:.0%} No DR)" for c, r in by_camera.iterrows()])
axes[0].invert_yaxis()
fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, labelcolor=INK_SOFT, ncols=2,
           loc="lower center", bbox_to_anchor=(0.5, 0.0))
fig.suptitle("Test split, camera by camera: final model against the camera-only model",
             fontsize=12, fontweight="bold", color=INK)
plt.tight_layout(rect=(0, 0.05, 1, 0.94))
plt.savefig(FIG_DIR / "fig31_test_by_camera.png")
plt.show()

# the camera whose images give the least away (44% healthy): the fairest single test of the model
mixed = (predictions["camera_group"] == config["mixed_camera"]).to_numpy()
mixed_scores = None
if mixed.sum() >= 5:
    mixed_scores = pd.DataFrame({"final model": full_scores(true[mixed], pred[mixed]),
                                 "camera only": full_scores(true[mixed], camera_guess[mixed])})
    print(f"\nthe mixed camera {config['mixed_camera']} ({mixed.sum()} test images):")
    print(mixed_scores.loc[["accuracy", "macro_f1", "qwk", "dr_accuracy", "dr_sensitivity",
                            "dr_specificity"]].round(3).to_string())
else:
    print(f"\nonly {mixed.sum()} test images from {config['mixed_camera']} - too few to score")

### 8.4 Do dark or blurry images fail more often?

Brightness and sharpness are measured on the standardised image the model saw (variant A, inside the retina window, from notebook 2). The test images are split into thirds for each. Read this table with care: the thirds also differ in camera and in stage mix (shown as *share No DR*), and an easy, mostly healthy group scores well whatever its quality.

In [ ]:
stats = pd.read_csv(find_file("preprocess_stats.csv"),
                    usecols=["id_code", "A_mean_r", "A_mean_g", "A_mean_b", "A_sharpness"])
quality = predictions.merge(stats, on="id_code")
quality["brightness"] = 0.299 * quality["A_mean_r"] + 0.587 * quality["A_mean_g"] + 0.114 * quality["A_mean_b"]
quality["sharpness"] = quality["A_sharpness"]
rows = []
for measure in ("brightness", "sharpness"):
    # rank first, so images with identical values cannot break the split into thirds
    thirds = pd.qcut(quality[measure].rank(method="first"), 3, labels=["lowest third", "middle third", "highest third"])
    for level, g in quality.groupby(thirds, observed=True):
        rows.append({"measure": measure, "level": level, "images": len(g),
                     "range": f"{g[measure].min():.3g}-{g[measure].max():.3g}",
                     "share_no_dr": (g["true"] == 0).mean(), "accuracy": accuracy_score(g["true"], g["pred"]),
                     "dr_accuracy": accuracy_score(g["true"] > 0, g["pred"] > 0),
                     "off_by_2_or_more": (np.abs(g["pred"] - g["true"]) >= 2).mean()})
by_quality = pd.DataFrame(rows)
by_quality.to_csv(OUT_DIR / "test_by_quality.csv", index=False)
print(by_quality.round(3).to_string(index=False))

### 8.5 The most serious mistakes

The test images the model got most wrong: the largest distance between the true stage and the answer first, then the most confident. Each image is shown as the model saw it (the chosen preprocessing variant). Looking at them helps separate model errors from hard or doubtful cases — a poorly exposed photo, a lesion at the very edge, or a label that looks questionable. Each APTOS image was graded by one clinician, and the organisers themselves warn that the labels contain noise.

In [ ]:
wrong = predictions[predictions["true"] != predictions["pred"]].copy()
wrong["confidence"] = probs[wrong.index.to_numpy(), wrong["pred"].to_numpy()]    # P(the answered stage)
wrong["distance"] = (wrong["pred"] - wrong["true"]).abs()
worst = wrong.sort_values(["distance", "confidence"], ascending=False).head(8)

if len(worst):
    shown = load_images(worst["id_code"], size=None)             # 512 px, before the resize to the model size
    columns = min(4, len(worst))
    rows_n = int(np.ceil(len(worst) / columns))
    fig, axes = plt.subplots(rows_n, columns, figsize=(3.3 * columns, 4.1 * rows_n), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, img, (_, r) in zip(axes.ravel(), shown, worst.iterrows()):
        ax.imshow(img)
        ax.set_title(f"true {int(r.true)} {CLASS_NAMES[int(r.true)]} → answer {int(r.pred)} {CLASS_NAMES[int(r.pred)]}\n"
                     f"P(DR) {r.p_dr:.2f} | P(answer) {r.confidence:.2f}", fontsize=8.5, color=INK_SOFT)
    fig.suptitle(f"The {len(worst)} most serious test mistakes (variant {VARIANT})",
                 fontsize=12, fontweight="bold", color=INK)
    plt.tight_layout(rect=(0, 0, 1, 0.95), h_pad=2.0)
    plt.savefig(FIG_DIR / "fig32_test_mistakes.png")
    plt.show()
print(worst[["id_code", "camera_group", "true", "pred", "p_dr", "confidence"]].round(3).to_string(index=False))

## 9. Grad-CAM: where does the model look?

**Grad-CAM** *(in your transfer-learning practical and the lecturer's notebook, where it failed with "No convolutional layer found")* colours the parts of an image that pushed the model towards its answer: red = strong evidence, blue = little. It works here because notebook 3 built the backbone "flat", so its last convolution layer can be reached.

The code lives in its own file, **`dr_gradcam.py`**, written by the next cell. The Streamlit app uses the same file, so the app's heatmaps are made in exactly the same way. How it works, in five steps:

1. Take the feature maps of the last convolution layer — 7 × 7 maps for a 224 px input (10 × 10 at 300 px).
2. Measure how much the answered stage's score would change if each map grew a little (its gradient).
3. Average the gradients over each map: one importance weight per map.
4. Add the maps up with those weights and keep only the positive part (evidence *for* the stage).
5. Scale to 0–1 and enlarge to the image size.

**Limits.** Each cell of a 7 × 7 map covers 32 × 32 pixels, so the heat is coarse. It can sit next to a lesion rather than exactly on it. For an answer of *No DR*, the map shows what the model relied on to call the eye healthy, which rarely points at anything specific. Grad-CAM shows where the model looked, not *why*, and a plausible-looking map does not prove the answer is right.

In [ ]:
%%writefile dr_gradcam.py
"""
dr_gradcam.py - Grad-CAM heatmaps for the DR model.

It is used in two places, so the heatmaps are made in exactly the same way:
  * Kaggle notebook 4 (dr-04-evaluate), for the error analysis;
  * the Streamlit prototype (app/), to explain the prediction for an uploaded photo.

Grad-CAM (Selvaraju et al., 2017) shows which parts of an image pushed the model towards one stage:
  1. take the feature maps of the last convolution layer (7 x 7 maps for a 224 px input);
  2. measure how much the stage's score would change if each map grew a little (its gradient);
  3. average those gradients over each map -> one importance weight per map;
  4. add the maps up with those weights and keep only the positive part (evidence FOR the stage);
  5. scale to 0-1 and enlarge to the image size, so it can be laid over the photo.
The score used is the stage's value before softmax (its "logit"), as in the original paper.

Computer Vision CW1 - Diabetic Retinopathy Stage Detection - Mohamed Shafran
"""
import numpy as np
import cv2
import keras
import tensorflow as tf

# the last convolution block of each backbone (the names Keras gives them)
LAST_CONV = {"EfficientNetB0": "top_activation", "EfficientNetB3": "top_activation",
             "MobileNetV2": "out_relu"}


class GradCam:
    """Grad-CAM for a model built by dr_training.build_model:
    image -> augmentation -> backbone -> 'gap' -> 'dropout' -> 'stage_probs' (Dense, softmax)."""

    def __init__(self, model, last_conv):
        self.model = model
        # one pass gives both the last feature maps and the pooled features that feed the output layer
        self.inner = keras.Model(model.inputs,
                                 [model.get_layer(last_conv).output, model.get_layer("gap").output])
        output_layer = model.get_layer("stage_probs")
        self.kernel, self.bias = output_layer.kernel, output_layer.bias

    def maps(self, images, stages):
        """Low-resolution heatmaps (N x h x w, each scaled to 0-1) for a batch of 0-255 images
        and, for each image, the stage to explain. Also returns the five probabilities."""
        x = tf.convert_to_tensor(np.asarray(images, dtype="float32"))
        stages = np.asarray(stages, dtype="int32")
        with tf.GradientTape() as tape:
            features_maps, pooled = self.inner(x, training=False)       # training=False: no augmentation
            logits = keras.ops.matmul(pooled, self.kernel) + self.bias  # scores before softmax
            score = tf.gather(logits, stages, axis=1, batch_dims=1)     # each image's own stage
        grads = tape.gradient(score, features_maps)                     # N x h x w x channels
        weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)     # step 3: one weight per map
        cams = tf.nn.relu(tf.reduce_sum(features_maps * weights, axis=-1)).numpy()   # step 4
        peak = cams.reshape(len(cams), -1).max(axis=1)
        cams = cams / np.maximum(peak, 1e-12)[:, None, None]            # step 5: 0-1 per image
        probs = keras.ops.convert_to_numpy(keras.ops.softmax(logits, axis=-1))
        return cams, probs

    def heatmap(self, image, stage, size=None):
        """One image: the heatmap enlarged to `size` x `size` (default: the image's own size)."""
        cams, _ = self.maps(image[None], [stage])
        return enlarge(cams[0], size or image.shape[0])


def enlarge(cam, size):
    """Smoothly enlarge a low-resolution heatmap (e.g. 7 x 7) to size x size."""
    return np.clip(cv2.resize(cam.astype(np.float32), (size, size), interpolation=cv2.INTER_CUBIC), 0, 1)


def overlay(image, cam, alpha=0.4):
    """Colour the heatmap (blue = low, red = high) and blend it over the image (RGB uint8)."""
    if cam.shape[:2] != image.shape[:2]:
        cam = enlarge(cam, image.shape[0])
    colour = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return np.uint8(np.clip((1 - alpha) * image.astype(np.float32) + alpha * colour, 0, 255))


In [ ]:
import dr_gradcam as G
importlib.reload(G)                        # picks up changes if the cell above is edited and re-run
cam = G.GradCam(model, G.LAST_CONV[BACKBONE])

# heatmaps for every test image, each for the stage the system answered (batches of 32)
maps = np.concatenate([cam.maps(X_test[i:i + 32], pred[i:i + 32])[0] for i in range(0, len(X_test), 32)])
print("heatmap size:", maps.shape[1:], "for", len(maps), "test images")

# Does the heat fall on the retina, or on the black background and the edge of the window?
window = cv2.resize(prep.MASK.astype(np.uint8), (SIZE, SIZE), interpolation=cv2.INTER_NEAREST).astype(bool)
inside = np.array([G.enlarge(m, SIZE)[window].sum() / max(G.enlarge(m, SIZE).sum(), 1e-9) for m in maps])
print(f"share of the heat inside the retina window: median {np.median(inside):.2f} "
      f"(a map spread evenly over the image would give {window.mean():.2f})")
for label, rows_mask in [("answered DR", pred > 0), ("answered No DR", pred == 0),
                         ("correct", pred == true), ("wrong", pred != true)]:
    if rows_mask.any():
        print(f"   {label:<15} median {np.median(inside[rows_mask]):.2f}  ({rows_mask.sum()} images)")

In [ ]:
def gradcam_panel(rows, title, path):
    """Two rows per example: the image the model saw (top) and the Grad-CAM overlay (bottom)."""
    shown = load_images(rows["id_code"], size=None)
    fig, axes = plt.subplots(2, len(rows), figsize=(3 * len(rows), 6.6), squeeze=False)
    for k, (img, (i, r)) in enumerate(zip(shown, rows.iterrows())):
        heat = G.enlarge(maps[i], img.shape[0])
        axes[0, k].imshow(img)
        axes[1, k].imshow(G.overlay(img, heat))
        axes[0, k].set_title(f"true {int(r.true)} {CLASS_NAMES[int(r.true)]}\nanswer {int(r.pred)} "
                             f"{CLASS_NAMES[int(r.pred)]} (P {probs[i, int(r.pred)]:.2f})", fontsize=8.5, color=INK_SOFT)
        for ax in axes[:, k]:
            ax.axis("off")
    fig.suptitle(title, fontsize=12, fontweight="bold", color=INK)
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    plt.savefig(path)
    plt.show()


# Figure 33 - one correct answer per stage: the TYPICAL one (median confidence), not the best
typical = []
for stage in range(5):
    right = predictions[(predictions["true"] == stage) & (predictions["pred"] == stage)]
    if len(right):
        confidence = probs[right.index.to_numpy(), stage]
        typical.append(right.index[np.argsort(confidence)[len(confidence) // 2]])
if typical:
    gradcam_panel(predictions.loc[typical], "Grad-CAM, correct answers (a typical example of each stage)",
                  FIG_DIR / "fig33_gradcam_correct.png")
missing = [CLASS_NAMES[s] for s in range(5) if s not in set(predictions.loc[typical, "true"])]
if missing:
    print("no correct answer to show for:", missing)

# Figure 34 - the five most serious mistakes from section 8.5
if len(worst):
    gradcam_panel(worst.head(5), "Grad-CAM, the most serious mistakes (map for the stage answered)",
                  FIG_DIR / "fig34_gradcam_mistakes.png")

## 10. Save the results

- **`test_predictions.csv`** — every test image: true stage, answer, the five probabilities, camera.
- **`test_results.csv`**, **`test_bootstrap_ci.csv`**, **`test_classification_report.csv`**, **`test_per_stage.csv`**, **`test_by_camera.csv`**, **`test_by_quality.csv`** — the tables above.
- **`test_metrics.json`** — the headline numbers in one file.
- **`dr_gradcam.py`** — the Grad-CAM code, also used by the app.
- **`report_bundle_nb4.zip`** — all of the above plus figures 26–34, for the report. **Download this file.**

In [ ]:
def clean(value):
    """JSON cannot store NumPy numbers or NaN; turn them into plain numbers or null."""
    if isinstance(value, dict):
        return {str(k): clean(v) for k, v in value.items()}
    if isinstance(value, (np.floating, float)):
        return None if np.isnan(value) else round(float(value), 4)
    if isinstance(value, np.integer):
        return int(value)
    return value

test_metrics = clean({
    "model": {k: config[k] for k in ("name", "config", "backbone", "variant", "balance", "fine_tune_lr",
                                     "image_size", "best_epoch")},
    "test_images": len(test),
    "test_images_per_stage": dict(zip(CLASS_NAMES, np.bincount(true, minlength=5).tolist())),
    "validation_check": {"identical_answers": same, **now},
    "results": results.to_dict(),
    "bootstrap_95ci": ci_table.to_dict(orient="index"),
    "mixed_camera": {"camera": config["mixed_camera"], "images": int(mixed.sum()),
                     **(mixed_scores.to_dict() if mixed_scores is not None else {})},
    "mistakes": {"total": wrong_total, "one_stage_away": one_away},
    "gradcam_share_inside_window": {"median": float(np.median(inside)), "uniform_map": float(window.mean())},
})
with open(OUT_DIR / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

with zipfile.ZipFile(OUT_DIR / "report_bundle_nb4.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(FIG_DIR.glob("*.png")):
        bundle.write(path, f"figures/{path.name}")
    for name in ("test_predictions.csv", "test_results.csv", "test_bootstrap_ci.csv",
                 "test_classification_report.csv", "test_per_stage.csv", "test_by_camera.csv",
                 "test_by_quality.csv", "test_metrics.json", "dr_gradcam.py"):
        bundle.write(OUT_DIR / name, name)
print(json.dumps(test_metrics["results"]["final model (test)"], indent=2))
print(f"\nreport_bundle_nb4.zip: {(OUT_DIR / 'report_bundle_nb4.zip').stat().st_size / 1e6:.1f} MB")

## 11. What to do next

1. Download **`report_bundle_nb4.zip`** and unzip it into `CW1_DR_Stage_Detection/results/notebook4/`.
2. Download **`best_model.keras`** from notebook 3's output into `CW1_DR_Stage_Detection/models/`, and copy `dr_gradcam.py` into `app/`. The Streamlit prototype uses both.
3. Send back `test_metrics.json`, `test_by_camera.csv` and figures 26–34, so the results can be checked and interpreted together.